In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

**High Frequency Bot Detection**

In [0]:
df = spark.table("default.silver_streams")
high_freq = df.groupBy("user_id", "song_id") \
    .agg(count("*").alias("metric_value")) \
    .filter("metric_value > 500") \
    .withColumn("fraud_type", lit("High_Frequency"))


**Short Play Abuse**

In [0]:
short_play = df.groupBy("user_id") \
    .agg(avg("duration_played").alias("metric_value")) \
    .filter("metric_value < 10") \
    .withColumn("song_id", lit(None)) \
    .withColumn("fraud_type", lit("Short_Duration"))

**Spike Detection**

In [0]:
time_fraud = df.groupBy(
    "user_id",
    window("timestamp", "1 hour")
).agg(count("*").alias("metric_value")) \
 .filter("metric_value > 100") \
 .withColumn("song_id", lit(None)) \
 .withColumn("fraud_type", lit("Time_Based")) \
 .select("user_id", "song_id", "metric_value", "fraud_type")


In [0]:
high_freq_final = high_freq.select("user_id", "song_id", "metric_value", "fraud_type")
short_play_final = short_play.select("user_id", "song_id", "metric_value", "fraud_type")

In [0]:
final_fraud = high_freq_final.unionByName(short_play_final) \
                             .unionByName(time_fraud)

In [0]:
final_fraud.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.fraud_detection_summary")

In [0]:
%sql
SELECT * FROM default.fraud_detection_summary;

user_id,song_id,metric_value,fraud_type
9999,111,1000.0,High_Frequency
9999,null,1000.0,Time_Based


**FRAUD DETECTION RAW**

In [0]:
df = spark.table("default.silver_streams")

In [0]:
window_user_song = Window.partitionBy("user_id", "song_id")

fraud_high_freq = df.withColumn(
    "metric_value",
    count("*").over(window_user_song)
).filter(col("metric_value") > 500) \
 .withColumn("fraud_type", lit("High_Frequency"))

In [0]:
window_user = Window.partitionBy("user_id")

fraud_low_duration = df.withColumn(
    "metric_value",
    avg("duration_played").over(window_user)
).filter(col("metric_value") < 10) \
 .withColumn("fraud_type", lit("Low_Avg_Duration"))

In [0]:
df = df.withColumn("ts_long", col("timestamp").cast("long"))

window_hour = Window.partitionBy("user_id") \
    .orderBy("ts_long") \
    .rangeBetween(-3600, 0)

fraud_high_hour = df.withColumn(
    "metric_value",
    count("*").over(window_hour)
).filter(col("metric_value") > 100) \
 .withColumn("fraud_type", lit("High_Hourly_Activity"))

In [0]:
common_columns = [
    "user_id",
    "song_id",
    "timestamp",
    "duration_played",
    "metric_value",
    "fraud_type"
]

In [0]:
fraud_high_freq = fraud_high_freq.select(common_columns)
fraud_low_duration = fraud_low_duration.select(common_columns)
fraud_high_hour = fraud_high_hour.select(common_columns)

In [0]:
fraud_raw = fraud_high_freq \
    .unionByName(fraud_low_duration) \
    .unionByName(fraud_high_hour) \
    .dropDuplicates()

In [0]:
fraud_raw.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.fraud_detection_raw")

In [0]:
%sql
SELECT * FROM default.fraud_detection_raw;

user_id,song_id,timestamp,duration_played,metric_value,fraud_type
9999,111,2026-03-02T12:22:25.401Z,297,1000.0,High_Frequency
9999,111,2026-03-02T12:22:25.403Z,227,1000.0,High_Frequency
9999,111,2026-03-02T12:22:25.425Z,292,1000.0,High_Frequency
9999,111,2026-03-02T12:22:25.446Z,205,1000.0,High_Frequency
9999,111,2026-03-02T12:22:25.473Z,222,1000.0,High_Frequency
9999,111,2026-03-02T12:22:25.645Z,223,1000.0,High_Frequency
9999,111,2026-03-02T12:22:25.682Z,205,1000.0,High_Frequency
9999,111,2026-03-02T12:22:25.696Z,267,1000.0,High_Frequency
9999,111,2026-03-02T12:22:25.700Z,200,1000.0,High_Frequency
9999,111,2026-03-02T12:22:25.748Z,248,1000.0,High_Frequency


In [0]:
gold = df.groupBy("song_id") \
    .agg(
        count("*").alias("total_streams"),
        countDistinct("user_id").alias("unique_users")
    )

In [0]:
gold.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("default.gold_kpis")

In [0]:
%sql
SELECT * FROM default.gold_kpis;

song_id,total_streams,unique_users
93,33,33
28,27,26
171,32,31
30,30,28
151,30,30
200,29,26
33,20,19
29,35,35
129,20,20
9,28,27
